# 📖 Notebook 5: Cache Stampede & Hot Keys

Caching solves the read bottleneck, but it introduces new failure modes that can be just as dangerous. In this notebook, we'll explore the two most critical problems:

1. **Cache Stampede (Thundering Herd)** — many requests rebuild the same key at once
2. **Hot Keys** — one key gets hammered by millions of requests

## Learning Objectives

- Reproduce a cache stampede and see it overwhelm the database
- Implement request coalescing (single flight) to prevent stampedes
- Implement early refresh to avoid TTL-triggered stampedes
- Understand the hot key problem and mitigation strategies

In [ ]:
import psycopg2
import redis
import json
import time
import threading
import random
from concurrent.futures import ThreadPoolExecutor, as_completed

DB_CONFIG = {
    "host": "localhost", "port": 5432,
    "database": "caching_demo", "user": "demo", "password": "demo"
}
REDIS_CONFIG = {"host": "localhost", "port": 6379, "decode_responses": True}

def get_db():
    return psycopg2.connect(**DB_CONFIG)

r = redis.Redis(**REDIS_CONFIG)
r.flushdb()
print("✅ Connected and Redis cleared")

## 💥 The Cache Stampede Problem

A cache stampede happens when a popular cache entry expires and **many requests try to rebuild it at the same time**.

```
  Time ─────────────────────────────────────────────▶
                                    
  Cache:  [████████████ TTL=60s ████████████]  EXPIRED!
                                                  │
  Request 1: ─────────────────────────────────── miss → query DB
  Request 2: ─────────────────────────────────── miss → query DB
  Request 3: ─────────────────────────────────── miss → query DB
  ...                                              ↓
  Request 100: ───────────────────────────────── miss → query DB
                                                  │
                                            💥 DB overwhelmed!
```

Instead of 1 query, the database suddenly gets 100+ identical queries.

In [ ]:
# Let's reproduce a cache stampede

# Track database queries to see the stampede
db_query_log = []
db_query_lock = threading.Lock()

def expensive_query(product_id: int) -> dict:
    """Simulate an expensive database query that takes 200ms."""
    with db_query_lock:
        db_query_log.append(time.time())
    
    conn = get_db()
    cursor = conn.cursor()
    # Simulate expensive work with pg_sleep
    cursor.execute("SELECT pg_sleep(0.2)")
    cursor.execute(
        "SELECT id, name, price FROM products WHERE id = %s", (product_id,)
    )
    row = cursor.fetchone()
    conn.close()
    return {"id": row[0], "name": row[1], "price": float(row[2])}

def naive_get_product(product_id: int) -> dict:
    """Cache-aside WITHOUT stampede protection."""
    cache_key = f"product:{product_id}"
    cached = r.get(cache_key)
    if cached:
        return json.loads(cached)
    
    # Cache miss — every request goes to DB
    product = expensive_query(product_id)
    r.setex(cache_key, 60, json.dumps(product))
    return product

# Simulate stampede: 50 concurrent requests for the same product
r.flushdb()
db_query_log = []

print("💥 Simulating Cache Stampede (50 concurrent requests, no protection):")
print()

start = time.time()
with ThreadPoolExecutor(max_workers=50) as pool:
    futures = [pool.submit(naive_get_product, 42) for _ in range(50)]
    results = [f.result() for f in as_completed(futures)]
total_time = time.time() - start

print(f"   Total requests:     50")
print(f"   Database queries:   {len(db_query_log)}  ← should be 1, got {len(db_query_log)}!")
print(f"   Total time:         {total_time:.2f}s")
print()
print(f"💥 {len(db_query_log)} identical queries hit the database!")
print("   In production with 10,000 req/s, this could crash the database.")

## 🛡️ Solution 1: Request Coalescing (Single Flight)

The most effective stampede fix. When a cache miss happens, **only one request** rebuilds the cache. All other requests **wait** for that one result.

```
  Request 1: miss → LOCK → query DB → cache result → unlock
  Request 2: miss → wait for lock... → read from cache
  Request 3: miss → wait for lock... → read from cache
  ...
  Request 100: miss → wait for lock... → read from cache
```

Result: **1 database query** instead of 100.

In [ ]:
def coalesced_get_product(product_id: int) -> dict:
    """
    Cache-aside WITH request coalescing using Redis SETNX as a lock.
    Only one request rebuilds the cache; others wait.
    """
    cache_key = f"product:{product_id}"
    lock_key = f"lock:{cache_key}"
    
    # Step 1: Check cache
    cached = r.get(cache_key)
    if cached:
        return json.loads(cached)
    
    # Step 2: Try to acquire the rebuild lock
    # SETNX = "SET if Not eXists" — only one thread wins
    got_lock = r.set(lock_key, "1", nx=True, ex=10)  # lock expires in 10s (safety)
    
    if got_lock:
        # We won the lock — rebuild the cache
        try:
            product = expensive_query(product_id)
            r.setex(cache_key, 60, json.dumps(product))
            return product
        finally:
            r.delete(lock_key)  # release the lock
    else:
        # Someone else is rebuilding — wait and retry
        for _ in range(50):  # wait up to 5 seconds
            time.sleep(0.1)
            cached = r.get(cache_key)
            if cached:
                return json.loads(cached)
        
        # Fallback: if we waited too long, go to DB directly
        return expensive_query(product_id)

print("✅ coalesced_get_product() ready")
print("   Uses Redis SETNX as a distributed lock.")

In [ ]:
# Test: same 50 concurrent requests, but WITH coalescing

r.flushdb()
db_query_log = []

print("🛡️ Same test WITH request coalescing:")
print()

start = time.time()
with ThreadPoolExecutor(max_workers=50) as pool:
    futures = [pool.submit(coalesced_get_product, 42) for _ in range(50)]
    results = [f.result() for f in as_completed(futures)]
total_time = time.time() - start

print(f"   Total requests:     50")
print(f"   Database queries:   {len(db_query_log)}  ← only {len(db_query_log)}!")
print(f"   Total time:         {total_time:.2f}s")
print()

if len(db_query_log) <= 2:
    print("✅ Stampede prevented! Only 1-2 queries hit the database.")
    print("   The other 48-49 requests waited for the cache to be populated.")
else:
    print(f"⚠️ Some leakage ({len(db_query_log)} queries), but much better than 50!")

## 🛡️ Solution 2: Early Refresh (Probabilistic Expiration)

Instead of waiting for TTL to expire, **proactively refresh** the cache before it expires.  
Each request near the end of the TTL has a **small probability** of refreshing the cache early.

This spreads refreshes over time instead of everyone hitting at the exact same moment.

In [ ]:
import math

def early_refresh_get_product(product_id: int, ttl: int = 60, beta: float = 1.0) -> dict:
    """
    Cache-aside with probabilistic early refresh.
    As TTL approaches expiry, there's a growing chance the
    current request will proactively refresh the cache.
    """
    cache_key = f"product:{product_id}"
    
    cached = r.get(cache_key)
    if cached:
        remaining_ttl = r.ttl(cache_key)
        
        # Probabilistic early refresh:
        # As remaining TTL gets small, probability of refresh increases.
        # Formula: refresh if remaining_ttl < beta * log(random())
        # This means ~1 request will refresh just before expiry.
        if remaining_ttl > 0:
            delta = ttl - remaining_ttl  # time since cached
            # probability increases as we approach expiry
            if delta > 0 and remaining_ttl < beta * (-math.log(random.random())) * 10:
                # Early refresh! Fetch from DB and update cache
                product = expensive_query(product_id)
                r.setex(cache_key, ttl, json.dumps(product))
                return product
        
        return json.loads(cached)
    
    # Normal cache miss
    product = expensive_query(product_id)
    r.setex(cache_key, ttl, json.dumps(product))
    return product

# Demo: watch early refresh in action
r.flushdb()
db_query_log = []

print("🔄 Early Refresh Demo (TTL=5s):")
print()

# Initial cache
product = early_refresh_get_product(42, ttl=5)
print(f"t=0s: Cached product 42 (TTL=5s, DB queries: {len(db_query_log)})")

# Read over time — watch for early refresh
for t in range(1, 8):
    time.sleep(1)
    queries_before = len(db_query_log)
    product = early_refresh_get_product(42, ttl=5)
    queries_after = len(db_query_log)
    
    remaining = r.ttl("product:42")
    refreshed = "🔄 REFRESHED" if queries_after > queries_before else "⚡ cache hit"
    print(f"t={t}s: {refreshed} (TTL remaining: {remaining}s, Total DB queries: {queries_after})")

print()
print("💡 Early refresh prevents stampedes by refreshing before TTL expires.")
print("   Only ~1 request pays the cost; everyone else gets a cache hit.")

## 🔥 The Hot Key Problem

A **hot key** is a cache entry that gets far more traffic than everything else. Even though caching works, one key can overwhelm a single Redis node.

Example: if everyone is viewing Taylor Swift's profile, the key `user:taylorswift` might get millions of requests per second — more than one Redis shard can handle.

### Solutions:
1. **Replicate the hot key** across multiple keys to spread load
2. **Add a local in-process cache** as a first layer

In [ ]:
r.flushdb()

# Solution 1: Key replication — spread one key across multiple replicas

NUM_REPLICAS = 5

def set_replicated(key: str, value: str, ttl: int = 60):
    """Write the same value to multiple replica keys."""
    for i in range(NUM_REPLICAS):
        r.setex(f"{key}:replica:{i}", ttl, value)

def get_replicated(key: str) -> str:
    """Read from a random replica to spread load."""
    replica = random.randint(0, NUM_REPLICAS - 1)
    return r.get(f"{key}:replica:{replica}")

# Demo: simulate hot key with replication
hot_product = {"id": 1, "name": "Viral Product", "price": 29.99}
set_replicated("product:1", json.dumps(hot_product))

# Simulate 1000 reads — track which replicas are hit
replica_hits = {i: 0 for i in range(NUM_REPLICAS)}

for _ in range(1000):
    replica = random.randint(0, NUM_REPLICAS - 1)
    r.get(f"product:1:replica:{replica}")
    replica_hits[replica] += 1

print("🔥 Hot Key Replication Demo (1000 reads, 5 replicas):")
print()
for i, hits in replica_hits.items():
    bar = "█" * (hits // 20)
    print(f"   Replica {i}: {hits:>4} hits  {bar}")
print()
print(f"   Without replication: 1 node handles all 1000 requests.")
print(f"   With replication:    each node handles ~{1000//NUM_REPLICAS} requests.")
print()
print("💡 In a Redis Cluster, each replica key can land on a different shard,")
print("   spreading the load across multiple machines.")

In [ ]:
# Solution 2: Local in-process cache for extremely hot keys

class TwoLayerCache:
    """
    Two-layer cache:
    Layer 1: Local Python dict (in-process, fastest)
    Layer 2: Redis (shared across servers)
    Layer 3: PostgreSQL (source of truth)
    """
    
    def __init__(self, local_ttl: float = 1.0):
        self.local_cache = {}        # key → (value, expires_at)
        self.local_ttl = local_ttl   # local cache TTL in seconds
        self.r = redis.Redis(**REDIS_CONFIG)
        self.stats = {"local_hits": 0, "redis_hits": 0, "db_hits": 0}
    
    def get(self, product_id: int) -> dict:
        cache_key = f"product:{product_id}"
        
        # Layer 1: Check local cache
        if cache_key in self.local_cache:
            value, expires_at = self.local_cache[cache_key]
            if time.time() < expires_at:
                self.stats["local_hits"] += 1
                return value
            del self.local_cache[cache_key]  # expired
        
        # Layer 2: Check Redis
        cached = self.r.get(cache_key)
        if cached:
            self.stats["redis_hits"] += 1
            product = json.loads(cached)
            # Promote to local cache
            self.local_cache[cache_key] = (product, time.time() + self.local_ttl)
            return product
        
        # Layer 3: Database
        self.stats["db_hits"] += 1
        conn = get_db()
        cursor = conn.cursor()
        cursor.execute("SELECT id, name, price FROM products WHERE id = %s", (product_id,))
        row = cursor.fetchone()
        conn.close()
        
        product = {"id": row[0], "name": row[1], "price": float(row[2])}
        
        # Store in both layers
        self.r.setex(cache_key, 60, json.dumps(product))
        self.local_cache[cache_key] = (product, time.time() + self.local_ttl)
        
        return product

# Demo: heavy reads on one product
r.flushdb()
cache = TwoLayerCache(local_ttl=2.0)

print("🏗️ Two-Layer Cache Demo (5000 reads on same product):")
print()

start = time.time()
for _ in range(5000):
    cache.get(42)
elapsed = time.time() - start

total = sum(cache.stats.values())
print(f"   Local cache hits:  {cache.stats['local_hits']:>5}  ({cache.stats['local_hits']/total*100:.1f}%)")
print(f"   Redis hits:        {cache.stats['redis_hits']:>5}  ({cache.stats['redis_hits']/total*100:.1f}%)")
print(f"   Database hits:     {cache.stats['db_hits']:>5}  ({cache.stats['db_hits']/total*100:.1f}%)")
print(f"   Total time:        {elapsed:.3f}s")
print()
print("💡 The local cache absorbs nearly all requests.")
print("   Redis is barely touched. The database sees almost nothing.")
print("   Trade-off: local cache is NOT shared across servers,")
print("   so each server has its own copy (can be briefly stale).")

## 📊 Putting It All Together

Let's build a production-ready cache that handles stampedes and hot keys.

In [ ]:
r.flushdb()
db_query_log = []

class ProductionCache:
    """
    Production-ready cache combining all patterns:
    - Cache-aside with TTL
    - Request coalescing (stampede prevention)
    - Local cache layer (hot key protection)
    """
    
    def __init__(self, ttl: int = 60, local_ttl: float = 1.0):
        self.r = redis.Redis(**REDIS_CONFIG)
        self.ttl = ttl
        self.local_cache = {}
        self.local_ttl = local_ttl
        self.stats = {"local": 0, "redis": 0, "db": 0, "coalesced": 0}
    
    def get(self, product_id: int) -> dict:
        cache_key = f"product:{product_id}"
        
        # Layer 1: Local cache
        if cache_key in self.local_cache:
            val, exp = self.local_cache[cache_key]
            if time.time() < exp:
                self.stats["local"] += 1
                return val
        
        # Layer 2: Redis
        cached = self.r.get(cache_key)
        if cached:
            self.stats["redis"] += 1
            product = json.loads(cached)
            self.local_cache[cache_key] = (product, time.time() + self.local_ttl)
            return product
        
        # Layer 3: DB with coalescing
        lock_key = f"lock:{cache_key}"
        got_lock = self.r.set(lock_key, "1", nx=True, ex=10)
        
        if got_lock:
            try:
                self.stats["db"] += 1
                product = expensive_query(product_id)
                self.r.setex(cache_key, self.ttl, json.dumps(product))
                self.local_cache[cache_key] = (product, time.time() + self.local_ttl)
                return product
            finally:
                self.r.delete(lock_key)
        else:
            # Wait for another request to rebuild
            self.stats["coalesced"] += 1
            for _ in range(50):
                time.sleep(0.1)
                cached = self.r.get(cache_key)
                if cached:
                    product = json.loads(cached)
                    self.local_cache[cache_key] = (product, time.time() + self.local_ttl)
                    return product
            return expensive_query(product_id)

# Grand finale: 100 concurrent requests with full protection
cache = ProductionCache(ttl=60, local_ttl=0.5)

print("🏆 Production Cache: 100 concurrent requests")
print("=" * 55)
print()

start = time.time()
with ThreadPoolExecutor(max_workers=100) as pool:
    futures = [pool.submit(cache.get, 42) for _ in range(100)]
    results = [f.result() for f in as_completed(futures)]
total_time = time.time() - start

print(f"   Local cache hits:   {cache.stats['local']:>4}")
print(f"   Redis hits:         {cache.stats['redis']:>4}")
print(f"   Database queries:   {cache.stats['db']:>4}  ← just {cache.stats['db']}!")
print(f"   Coalesced (waited): {cache.stats['coalesced']:>4}")
print(f"   Total time:         {total_time:.2f}s")
print(f"   All returned data:  {'✅ yes' if all(r is not None for r in results) else '❌ no'}")
print()
print("🏆 100 requests, ~1 database query. That's what good caching looks like!")

## 🧹 Cleanup

In [ ]:
r.flushdb()
print("🧹 Redis cleared")

## 📚 Summary

### Cache Stampede Prevention

| Technique | How It Works | Complexity |
|-----------|-------------|------------|
| **Request coalescing** | Lock + wait — only 1 request rebuilds | Medium |
| **Early refresh** | Probabilistic refresh before TTL expires | Low |
| **Cache warming** | Pre-populate cache before traffic arrives | Low |

### Hot Key Mitigation

| Technique | How It Works | Trade-off |
|-----------|-------------|------------|
| **Key replication** | Same value on multiple keys | More memory, write amplification |
| **Local cache layer** | In-process dict as L1 cache | Not shared across servers |
| **Rate limiting** | Throttle abusive traffic | May affect legitimate users |

### Key Takeaways

1. **Stampedes are real** — a popular key expiring can crash your database
2. **Request coalescing** (single flight) is the most effective stampede fix
3. **Hot keys** can overwhelm a single cache node even with high hit rates
4. **Two-layer caching** (local + Redis) handles extreme traffic
5. In interviews, mention stampede prevention when discussing caching — it shows depth

### 🎓 Series Complete!

You've now covered the complete caching landscape:
1. ✅ Why caching matters and where to cache
2. ✅ Cache-aside pattern (the default)
3. ✅ Write-through and write-behind (keeping writes in sync)
4. ✅ Cache invalidation and TTL (keeping data fresh)
5. ✅ Stampede prevention and hot keys (handling failure modes)

**For interviews**: start with cache-aside + TTL + invalidation. Mention stampede prevention and hot key mitigation to show depth. Don't over-engineer — simple caching solves most problems.